# RAG Strategy Comparison: Dense → Hybrid → Reranking → Advanced Generation

This notebook evaluates **6 progressive RAG configurations** on a medical QA corpus.
Each section runs a head-to-head comparison, with results used to select the best
configuration before proceeding to the next experiment.
| Section | Experiment | Hypothesis |
|---------|-----------|------------|
| 0 | Setup: load indexes, chunks, and all helper functions | — |
| 1 | RAG2: `section_400token` vs `fixed_500char` chunking | Section-aware chunking improves retrieval recall |
| 2 | RAG1 (dense-only) vs RAG2 (hybrid) | Hybrid outperforms dense-only on exact medical terms |
| 3 | RAG2 vs RAG3 (+ reranking) | Cross-encoder reranking improves context precision |
| 4 | RAG3 vs RAG4 (+ gradient selection) | Pruning low-salience context improves answer accuracy |
| 5 | RAG4 vs RAG5 (diverse prompts) vs RAG6 (k-means vote) | Compare advanced generation strategies |

---

## Output for Evaluation

Every RAG run produces a CSV with three columns that map directly to the
`medical-rag-eval.ipynb` loader:

| This notebook | `medical-rag-eval.ipynb` | Description |
|---|---|---|
| `input` | `input` | Original question |
| `actual_output` | `actual_output` | RAG-generated answer |
| `retrieval_context` | `retrieval_context` | Context passed to the generator |

Each section's results cell is followed by an **Export Eval CSVs** cell that writes
per-RAG eval files (e.g. `eval_s3_rag3.csv`). The Final Summary cell also writes
`eval_all_rags.csv` combining all variants. To evaluate any file:

```python
# In medical-rag-eval.ipynb — replace the csv_data block with:
with open("eval_s3_rag3.csv", "r") as f:
    reader = csv.reader(f)
    next(reader)  # skip header
    for values in reader: ...
```

---
# Section 0: Setup — Imports, Configuration & Loading Chunks

Run all cells in this section **once** before any experiment. They define:
- Global paths and parameters
- All library imports
- Index download verification
- Retrievers for both chunking strategies
- All RAG helper functions shared across sections

## 0.1 — Global Configuration

Set paths and shared parameters here. `BEST_CHUNK_STRATEGY` starts as `None`
and is set automatically at the end of Section 1.

In [1]:
import os
from pathlib import Path

CWD = Path.cwd().resolve()


def _resolve_project_root(start: Path) -> Path:
    """Find the nearest parent that looks like the repo root."""
    for p in [start, *start.parents]:
        if (p / "src" / "data_loader.py").exists():
            return p
    return start


def _resolve_data_root(cwd: Path, project_root: Path) -> tuple[Path, str]:
    """Resolve data root with env override + robust auto-detection."""
    env_root = os.environ.get("RAG_DATA_ROOT")
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if p.exists():
            return p, "env:RAG_DATA_ROOT"
        print(f"[WARN] RAG_DATA_ROOT set but not found: {p}")

    candidates = []
    candidates.append(project_root / "data")
    for p in [cwd, *cwd.parents]:
        candidates.append(p / "data")

    # Deduplicate while preserving order
    seen = set()
    unique_candidates = []
    for c in candidates:
        key = str(c)
        if key not in seen:
            seen.add(key)
            unique_candidates.append(c)

    def score(path: Path) -> int:
        points = 0
        if (path / "bm25_index" / "section_400token.pkl").exists():
            points += 5
        if (path / "bm25_index" / "fixed_500char.pkl").exists():
            points += 5
        if (path / "vector_store" / "section_400token").exists():
            points += 5
        if (path / "vector_store" / "fixed_500char").exists():
            points += 5
        if (path / "test_final.csv").exists():
            points += 2
        if (path / "bm25_index.zip").exists():
            points += 1
        if (path / "vector_store.zip").exists():
            points += 1
        return points

    existing = [c for c in unique_candidates if c.exists()]
    if existing:
        best = max(existing, key=score)
        if score(best) > 0:
            return best.resolve(), "auto-detected"

    return (project_root / "data").resolve(), "fallback:project_root/data"


PROJECT_ROOT = _resolve_project_root(CWD)
DATA_ROOT, DATA_ROOT_SOURCE = _resolve_data_root(CWD, PROJECT_ROOT)

CSV_PATH = CWD / "test_final.csv" if (CWD / "test_final.csv").exists() \
           else DATA_ROOT / "test_final.csv"

VECTOR_DIR = DATA_ROOT / "vector_store"
BM25_DIR = DATA_ROOT / "bm25_index"
RAG_DIR = DATA_ROOT / "rag_output"
RAG_DIR.mkdir(parents=True, exist_ok=True)

N_QUESTIONS   = 50    # max 300
TOP_K_CONTEXT = 5    # final context chunks passed to generator
REFERENCE_COL = "ideal_answer"  # ground-truth column in test CSV

# RAG5 / RAG6 generation params
N_GENERATIONS = 7
TEMPERATURE   = 0.8
N_CLUSTERS    = 3

# Set automatically at the end of Section 1
BEST_CHUNK_STRATEGY = None
dense_fn_best       = None
bm25_fn_best        = None

print("CWD        :", CWD)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT  :", DATA_ROOT, f"({DATA_ROOT_SOURCE})")
print("CSV_PATH   :", CSV_PATH, "| exists?", CSV_PATH.exists())

CWD        : /workspace/notebooks
PROJECT_ROOT: /workspace
DATA_ROOT  : /workspace/data (auto-detected)
CSV_PATH   : /workspace/data/test_final.csv | exists? True


## 0.2 — Imports

In [2]:
import sys, pickle
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ── Resolve src/ ──────────────────────────────────────────────────────────────
_cwd = Path.cwd().resolve()
for _ in range(5):
    if (_cwd / "src" / "data_loader.py").exists():
        sys.path.insert(0, str(_cwd / "src"))
        print("Using src:", _cwd / "src")
        break
    _cwd = _cwd.parent
else:
    raise FileNotFoundError("Could not find src/ with data_loader.py")

from data_loader import load_and_sample_test_set
from answer import format_context, generate_answer, extract_pmids
from bm25 import spacy_tokenize_texts          # must match tokeniser used at index build time
from chroma_manager import ChromaManager       # wraps chromadb + PubMedBERT
from hybrid import hybrid_retrieve

print("All imports loaded successfully.")

/opt/conda/lib/python3.10/site-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Using src: /workspace/src


2026-03-20 09:32:32.689332325 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


All imports loaded successfully.


## 0.3 — Index Download & Verification

Download `bm25_index.zip` and `vector_store.zip` from Google Drive and place them in `data/`:

```
https://drive.google.com/drive/u/1/folders/1GsSlv4QWTTcBjnaZ8q2Dor8wsdfDwSDh
```

Then extract:
```bash
unzip bm25_index.zip -d data/
unzip vector_store.zip -d data/
```

Expected layout after extraction:
```
data/
├── bm25_index/
│   ├── section_400token.pkl
│   ├── fixed_500char.pkl
│   └── section_contextual_400token.pkl
└── vector_store/
    ├── section_400token/pubmedbert/
    ├── fixed_500char/pubmedbert/
    └── section_contextual_400token/pubmedbert/
```

> Skip this cell if indexes are already extracted.

In [3]:
'''import zipfile

# ==============================
# Extract ZIP files
# ==============================
for zip_name in ["bm25_index.zip", "vector_store.zip"]:
    zp = DATA_ROOT / zip_name
    if zp.exists():
        print(f"Extracting {zip_name} ...")
        with zipfile.ZipFile(zp, "r") as zf:
            zf.extractall(DATA_ROOT)
        print("  Done.")
    else:
        print(f"  {zip_name} not found — skipping (already extracted?).")

# ==============================
# Expected files / folders
# ==============================
EXPECTED = [
    # BM25 indexes
    DATA_ROOT / "bm25_index" / "section_400token.pkl",
    DATA_ROOT / "bm25_index" / "fixed_500char.pkl",
    DATA_ROOT / "bm25_index" / "section_contextual_400token.pkl",  # ✅ NEW

    # Vector stores
    DATA_ROOT / "vector_store" / "section_400token",
    DATA_ROOT / "vector_store" / "fixed_500char",
    DATA_ROOT / "vector_store" / "section_contextual_400token",    # ✅ NEW
]

# ==============================
# Verification
# ==============================
print("\nIndex verification:")
all_ok = True

for p in EXPECTED:
    ok = p.exists()
    print(f"  [{'OK' if ok else 'MISSING'}] {p.relative_to(DATA_ROOT)}")
    if not ok:
        all_ok = False

# ==============================
# Error handling
# ==============================
if not all_ok:
    raise FileNotFoundError(
        "One or more indexes are missing.\n"
        "Download from: https://drive.google.com/drive/u/1/folders/1GsSlv4QWTTcBjnaZ8q2Dor8wsdfDwSDh"
    )

print("\nAll indexes present. Ready to proceed.")'''

'import zipfile\n\n# ==============================\n# Extract ZIP files\n# ==============================\nfor zip_name in ["bm25_index.zip", "vector_store.zip"]:\n    zp = DATA_ROOT / zip_name\n    if zp.exists():\n        print(f"Extracting {zip_name} ...")\n        with zipfile.ZipFile(zp, "r") as zf:\n            zf.extractall(DATA_ROOT)\n        print("  Done.")\n    else:\n        print(f"  {zip_name} not found — skipping (already extracted?).")\n\n# ==============================\n# Expected files / folders\n# ==============================\nEXPECTED = [\n    # BM25 indexes\n    DATA_ROOT / "bm25_index" / "section_400token.pkl",\n    DATA_ROOT / "bm25_index" / "fixed_500char.pkl",\n    DATA_ROOT / "bm25_index" / "section_contextual_400token.pkl",  # ✅ NEW\n\n    # Vector stores\n    DATA_ROOT / "vector_store" / "section_400token",\n    DATA_ROOT / "vector_store" / "fixed_500char",\n    DATA_ROOT / "vector_store" / "section_contextual_400token",    # ✅ NEW\n]\n\n# ==============

## 0.4 — Load Test Set & Build Retrievers for Both Chunk Strategies

All chunking strategies are initialised here so Section 1 can compare them without re-running setup.

For each strategy, `build_retrievers()` returns:
- `dense_fn(query, n)` — top-n results via PubMedBERT cosine similarity (`ChromaManager`)
- `bm25_fn(query, n)` — top-n results via BM25 (pre-built `.pkl` index, spaCy tokenisation)
- `chunks_df` — DataFrame of all chunks and metadata

| Strategy | Description |
|---|---|
| `section_400token`            | Section-aware chunks (~400 tokens). Preserves semantic unit boundaries.                                               |
| `fixed_500char`               | Fixed-length chunks (500 characters). May split mid-sentence.                                                         |
| `section_contextual_400token` | Section-aware chunks enriched with LLM-generated summaries. Improves semantic representation and retrieval alignment. |


In [4]:
# ── Load test set ─────────────────────────────────────────────────────────────
test_df = load_and_sample_test_set(CSV_PATH, n_total=N_QUESTIONS, random_state=42)
print("Sampled questions:", len(test_df))

ref_map = dict(zip(test_df["question"], test_df[REFERENCE_COL])) \
          if REFERENCE_COL in test_df.columns else {}


# ── Helper: safely fetch full Chroma collection in pages ───────────────────────
def _fetch_collection_records(collection, batch_size: int = 1000):
    """
    Fetch all ids/documents/metadatas from a Chroma collection in paged requests.

    This avoids SQLite parameter-limit errors that can occur with one large get().
    """
    total = collection.count()
    all_ids, all_docs, all_metas = [], [], []

    for offset in range(0, total, batch_size):
        batch = collection.get(
            include=["documents", "metadatas"],
            limit=batch_size,
            offset=offset,
        )

        batch_ids = batch.get("ids") or []
        batch_docs = batch.get("documents") or []
        batch_metas = batch.get("metadatas") or [{}] * len(batch_ids)

        if len(batch_metas) < len(batch_ids):
            batch_metas = batch_metas + [{}] * (len(batch_ids) - len(batch_metas))

        all_ids.extend(batch_ids)
        all_docs.extend(batch_docs)
        all_metas.extend(batch_metas)

    return all_ids, all_docs, all_metas


# ── Helper: initialise retriever pair for one chunk strategy ──────────────────
def build_retrievers(chunk_strategy: str):
    """
    Initialise and return (dense_fn, bm25_fn, chunks_df) for the given
    chunk strategy.

    Dense retrieval uses ChromaManager (PubMedBERT embeddings stored in Chroma).
    BM25 retrieval loads the pre-built pickle index and tokenises queries with spaCy,
    matching the tokeniser used at index build time.

    Args:
        chunk_strategy: 'section_400token' or 'fixed_500char'

    Returns:
        dense_fn  : callable(query: str, n: int) -> list[(chunk_id, chunk_text, meta)]
        bm25_fn   : callable(query: str, n: int) -> list[(chunk_id, chunk_text, meta)]
        chunks_df : pd.DataFrame with 'chunk_text' column and metadata
    """
    # Dense retriever via ChromaManager
    chroma_db = ChromaManager(
        base_directory=VECTOR_DIR,
        chunk_strategy=chunk_strategy,
        embedding_model="pubmedbert",
        collection_name="medical_rag"
    )

    # Build chunks_df from Chroma collection (paged read to avoid SQL var limits)
    ids, docs, metas = _fetch_collection_records(chroma_db.collection, batch_size=1000)

    def _key(i):
        m = metas[i] if i < len(metas) and isinstance(metas[i], dict) else {}
        v = m.get("chunk_idx", ids[i])
        if isinstance(v, (int, float)):
            return int(v)
        if isinstance(v, str) and v.isdigit():
            return int(v)
        return i

    order = sorted(range(len(ids)), key=_key)
    chunks_df = pd.DataFrame({"chunk_text": [docs[i] for i in order]})

    meta_keys = set()
    for m in metas:
        if isinstance(m, dict):
            meta_keys.update(m.keys())

    for k in sorted(meta_keys):
        chunks_df[k] = [
            metas[i].get(k) if i < len(metas) and isinstance(metas[i], dict) else None
            for i in order
        ]

    # Prebuild ordered metadata for fast top-n retrieval returns
    ordered_metas = []
    for i in order:
        m = metas[i] if i < len(metas) and isinstance(metas[i], dict) else {}
        ordered_metas.append(m)

    def dense_fn(q: str, n: int) -> list:
        """Return top-n (chunk_id, chunk_text, meta) via PubMedBERT cosine similarity."""
        res = chroma_db.query(q, n_results=n)

        docs_res = (res.get("documents") or [[]])[0]
        metas_res = (res.get("metadatas") or [[]])[0]
        ids_res = (res.get("ids") or [[]])[0]

        out = []
        for i, d in enumerate(docs_res):
            m = metas_res[i] if i < len(metas_res) and isinstance(metas_res[i], dict) else {}
            cid = m.get("chunk_idx")
            if cid is None:
                cid = ids_res[i] if i < len(ids_res) else i
            out.append((cid, d, m))
        return out

    # BM25 retriever — pre-built pickle, spaCy tokenisation
    with open(BM25_DIR / f"{chunk_strategy}.pkl", "rb") as f:
        bm25_index = pickle.load(f)

    def bm25_fn(q: str, n: int) -> list:
        """Return top-n (chunk_id, chunk_text, meta) via BM25 lexical scoring."""
        tok = spacy_tokenize_texts([q])[0]
        scores = bm25_index.get_scores(tok)
        top = np.argsort(scores)[-n:][::-1]

        out = []
        for idx in top.tolist():
            text = chunks_df["chunk_text"].iloc[idx]
            meta = ordered_metas[idx] if idx < len(ordered_metas) else {}
            cid = meta.get("chunk_idx", idx) if isinstance(meta, dict) else idx
            out.append((cid, text, meta if isinstance(meta, dict) else {}))
        return out

    return dense_fn, bm25_fn, chunks_df


# ==============================
# Initialise all strategies upfront
# ==============================

# Section-aware
print("Loading section_400token retrievers...")
dense_fn_section, bm25_fn_section, chunks_df_section = build_retrievers("section_400token")
print(f"  chunks: {len(chunks_df_section):,}")

# Fixed-length
print("Loading fixed_500char retrievers...")
dense_fn_fixed, bm25_fn_fixed, chunks_df_fixed = build_retrievers("fixed_500char")
print(f"  chunks: {len(chunks_df_fixed):,}")

# Contextual section-aware (NEW)
print("Loading section_contextual_400token retrievers...")
dense_fn_contextual, bm25_fn_contextual, chunks_df_contextual = build_retrievers("section_contextual_400token")
print(f"  chunks: {len(chunks_df_contextual):,}")

print("\nAll retrievers ready for all chunking strategies.")

Sampled questions: 50
Loading section_400token retrievers...

Loading embedding model: pritamdeka/S-PubMedBert-MS-MARCO


/opt/conda/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vector DB ready → /workspace/data/vector_store/section_400token/pubmedbert


Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


  chunks: 109,234
Loading fixed_500char retrievers...

Loading embedding model: pritamdeka/S-PubMedBert-MS-MARCO


...

  chunks: 176,626
Loading section_contextual_400token retrievers...

Loading embedding model: pritamdeka/S-PubMedBert-MS-MARCO


/opt/conda/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vector DB ready → /workspace/data/vector_store/section_contextual_400token/pubmedbert


Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


  chunks: 109,234

All retrievers ready for all chunking strategies.


In [5]:
print(bm25_fn_section)

<function build_retrievers.<locals>.bm25_fn at 0x7f14dc33aef0>


## 0.5 — RAG Helper Functions

All pipeline components are defined here and reused across sections:

| Function | Used in | Purpose |
|---|---|---|
| `score_results` | All sections | Exact-match accuracy against ground truth |
| `export_for_eval` | All sections | Write eval-ready CSV (`input`, `actual_output`, `retrieval_context`) for `medical-rag-eval.ipynb` |
| `compare` | All sections | Print head-to-head results table |
| `run_rag_loop` | All sections | Core evaluation loop |
| `rerank_chunks` | RAG3, RAG4, RAG5, RAG6 | Cross-encoder reranking |
| `gradient_select_chunks` | RAG4, RAG5, RAG6 | Gradient-based chunk pruning |
| `diverse_prompted_generate` | RAG5 | Multi-persona generation with token-overlap vote |
| `kmeans_majority_generate` | RAG6 | Multi-sample generation with K-Means centroid vote |

In [11]:
# ═══════════════════════════════════════════════════════════════════════════════
# Evaluation utilities
# ═══════════════════════════════════════════════════════════════════════════════

def score_results(df: pd.DataFrame, ref_map: dict) -> float:
    """
    Compute exact-match accuracy: fraction of predictions that exactly match
    the ground-truth answer (case-insensitive, whitespace-stripped).

    Operates on the 'actual_output' column — the same column name used by
    medical-rag-eval.ipynb, so DataFrames can be scored here and exported
    to the eval notebook without any column renaming.

    Returns float in [0, 1], or float('nan') if ref_map is empty.
    """
    if not ref_map:
        return float("nan")
    df = df.copy()
    df["reference"] = df["input"].map(ref_map)
    df["correct"]   = df.apply(
        lambda r: str(r["actual_output"]).strip().lower() ==
                  str(r["reference"]).strip().lower()
        if pd.notna(r.get("reference")) else None, axis=1
    )
    return float(df["correct"].mean())


def export_for_eval(df: pd.DataFrame, path: str) -> str:
    """
    Write a CSV that is directly loadable by the medical-rag-eval notebook.

    The eval notebook reads a CSV with exactly three columns:
        input             — the original question
        actual_output     — the RAG-generated answer
        retrieval_context — the context string passed to the generator

    This function selects those three columns (in order) from a run_rag_loop
    result DataFrame and saves them to `path`.

    Args:
        df   : DataFrame returned by run_rag_loop.
        path : Output CSV file path (e.g. 'eval_rag3.csv').

    Returns:
        Resolved absolute path string of the saved file.
    """
    out = df[["input", "actual_output", "retrieval_context"]].copy()
    out.to_csv(RAG_DIR /path, index=False, encoding="utf-8")
    resolved = str(Path(path).resolve())
    print(f"  Eval CSV saved → {resolved}")
    return resolved


def compare(results: dict) -> pd.DataFrame:
    """
    Print a formatted head-to-head comparison table and return it as a DataFrame.

    Args:
        results: {label: pd.DataFrame} where each DataFrame has columns
                 [input, actual_output, retrieval_context] (eval-notebook format).
    """
    rows = []
    for label, df in results.items():
        acc = score_results(df, ref_map)
        rows.append({
            "Variant": label,
            "n": len(df),
            "Exact Match Acc": f"{acc:.4f}" if not pd.isna(acc) else "N/A",
        })
    cmp_df = pd.DataFrame(rows)
    print(cmp_df.to_string(index=False))
    return cmp_df


print("Evaluation utilities loaded.")

Evaluation utilities loaded.


In [12]:
# ═══════════════════════════════════════════════════════════════════════════════
# Core RAG evaluation loop
# ═══════════════════════════════════════════════════════════════════════════════

def run_rag_loop(
    label: str,
    dense_fn,
    bm25_fn,
    chunk_strategy: str,
    use_bm25: bool      = True,
    use_reranking: bool = False,
    use_gradient: bool  = False,
    use_diverse: bool   = False,
    use_kmeans: bool    = False,
    output_csv: str     = None,
) -> pd.DataFrame:
    """
    Run retrieval + generation for one RAG configuration over the test set.

    Pipeline:
        Query
          → Dense retrieval (always, via ChromaManager/PubMedBERT)
          → BM25 retrieval  (if use_bm25; score-fused with dense via hybrid_retrieve)
          → Cross-encoder reranking (if use_reranking)
          → Gradient-based chunk pruning (if use_gradient)
          → Generation:
              K-Means majority vote   (if use_kmeans)
              Diverse persona prompts (elif use_diverse)
              Standard single prompt  (otherwise)

    Args:
        label          : Display name for this run (e.g. 'RAG3').
        dense_fn       : Dense retriever callable — (query, n) -> [(id, text)].
        bm25_fn        : BM25 retriever callable  — (query, n) -> [(id, text)].
        chunk_strategy : 'section_400token' or 'fixed_500char'.
        use_bm25       : Include BM25 in retrieval fusion.
        use_reranking  : Apply cross-encoder reranking after retrieval.
        use_gradient   : Apply gradient-based chunk pruning after reranking.
        use_diverse    : Use diverse persona prompts for generation (RAG5).
        use_kmeans     : Use K-Means majority vote for generation (RAG6).
        output_csv     : Optional path to save results CSV.

    Returns:
        pd.DataFrame with columns:
            input, generated_output, retrieved_context, rag_variant, chunk_strategy
    """
    n_pre_rerank = 10 if (use_reranking or use_gradient or use_diverse or use_kmeans) else 5
    rows = []

    for i, row in test_df.iterrows():
        q = row["question"]
        if i == 0 or (i + 1) % 10 == 0:
            print(f"  [{label}] {i+1}/{len(test_df)}")

        # Step 1 — Retrieval
        if use_bm25:
            chunks = hybrid_retrieve(
                q, dense_fn, bm25_fn,
                n_dense=80, n_bm25=80, n_final=n_pre_rerank
            )
            #print(chunks)
        else:
            chunks = dense_fn(q, n_pre_rerank)
            #print(chunks)

        # Step 2 — Reranking
        if use_reranking or use_gradient or use_diverse or use_kmeans:
            chunks = rerank_chunks(q, chunks, top_k=TOP_K_CONTEXT)
            
        # Step 3 — Gradient-based pruning
        if use_gradient or use_diverse or use_kmeans:
            chunks = gradient_select_chunks(q, chunks, retain_ratio=0.6)

        context = format_context(chunks, sep="\n\n---\n\n")

        pmids = extract_pmids(chunks)
        pmids_str = ";".join(pmids)

        # Step 4 — Generation
        if use_kmeans:
            answer = kmeans_majority_generate(
                q, context, n_generations=N_GENERATIONS,
                temperature=TEMPERATURE, n_clusters=N_CLUSTERS
            )
        elif use_diverse:
            answer = diverse_prompted_generate(
                q, context, n_generations=N_GENERATIONS, temperature=TEMPERATURE
            )
        else:
            answer = generate_answer(q, context, model_name="Qwen/Qwen2.5-3B-Instruct")

        rows.append({
            "input":              q,
            "actual_output":      answer,      # eval notebook expects 'actual_output'
            "retrieval_context":  context,     # eval notebook expects 'retrieval_context'
            "retrieved_pmids":    pmids_str,   # ✅ NEW
            "rag_variant":        label,
            "chunk_strategy":     chunk_strategy,
        })

    df = pd.DataFrame(rows)
    if output_csv:
        df.to_csv(RAG_DIR /output_csv, index=False, encoding="utf-8")
        print(f"  Saved → {Path(output_csv).resolve()}")
    return df


print("run_rag_loop loaded.")

run_rag_loop loaded.


In [13]:
# ═══════════════════════════════════════════════════════════════════════════════
# Reranker — used in RAG3, RAG4, RAG5, RAG6
# ═══════════════════════════════════════════════════════════════════════════════

RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
reranker = CrossEncoder(RERANKER_MODEL)

def rerank_chunks(question: str, chunks: list, top_k: int = 10) -> list:
    """
    Rerank (chunk_id, text) pairs using a cross-encoder.

    The cross-encoder scores each (query, chunk) pair jointly — query and document
    tokens attend to each other — giving more accurate relevance estimates than the
    bi-encoder used in dense retrieval. Applied only to top-N candidates (not the
    full corpus) due to latency.

    Args:
        question : User query string.
        chunks   : List of (chunk_id, chunk_text) tuples from retrieval.
        top_k    : Number of top-scoring chunks to return.

    Returns:
        List of (chunk_id, chunk_text) sorted by cross-encoder score (desc),
        truncated to top_k.
    """
    if not chunks:
        return []
    scores = reranker.predict([(question, c[1]) for c in chunks])
    return [c for c, _ in
            sorted(zip(chunks, scores), key=lambda x: x[1], reverse=True)[:top_k]]


print(f"Reranker loaded: {RERANKER_MODEL}")

/opt/conda/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Reranker loaded: cross-encoder/ms-marco-MiniLM-L-6-v2


In [14]:
# ═══════════════════════════════════════════════════════════════════════════════
# Gradient-Based Context Selection — used in RAG4, RAG5, RAG6
# ═══════════════════════════════════════════════════════════════════════════════

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

_GEN_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

_gen_tokenizer  = AutoTokenizer.from_pretrained(
    _GEN_MODEL_NAME,
    trust_remote_code=True
)

_gen_model      = AutoModelForCausalLM.from_pretrained(
    _GEN_MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

_gen_model.eval()


def gradient_select_chunks(
    question: str,
    chunks: list,
    retain_ratio: float = 0.6,
    sep: str = "\n\n---\n\n"
) -> list:
    """
    Filter context chunks by gradient-based saliency.

    Runs one forward+backward pass through the generator to compute per-token
    saliency scores:
        s_i = ||d(Loss)/d(embedding_i)||_2

    Chunks are ranked by mean token saliency. The bottom (1 - retain_ratio)
    fraction are discarded, leaving high-signal chunks for the generator.

    Trade-off: ~2x inference time vs. RAG3 due to the extra backward pass.
    Recommended retain_ratio: 0.5–0.7.

    Args:
        question     : Query string.
        chunks       : List of (chunk_id, chunk_text) tuples (post-reranking).
        retain_ratio : Fraction of chunks to keep (0.0–1.0).
        sep          : Separator used between chunks in the prompt.

    Returns:
        Filtered list of (chunk_id, chunk_text) in original order.
    """
    if len(chunks) <= 1:
        return chunks

    chunk_texts  = [c[1] for c in chunks]
    prompt       = f"Question: {question}\nContext: {sep.join(chunk_texts)}\nAnswer:"

    inputs = _gen_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048   # 🔥 adjusted for Qwen + speed
    ).to(_gen_model.device)

    input_ids    = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]

    # ==========================
    # Generate reference output
    # ==========================
    with torch.no_grad():
        ref_ids = _gen_model.generate(
            input_ids,
            max_new_tokens=32,
            do_sample=False
        )

    # ==========================
    # Get embeddings (Qwen-compatible)
    # ==========================
    embed_layer = _gen_model.get_input_embeddings()
    embeds = embed_layer(input_ids).detach().requires_grad_(True)

    # ==========================
    # Forward + backward
    # ==========================
    outputs = _gen_model(
        inputs_embeds=embeds,
        attention_mask=attention_mask,
        labels=input_ids   # causal LM loss
    )

    outputs.loss.backward()

    saliency = embeds.grad.norm(dim=-1).squeeze(0).detach().cpu().numpy()

    # ==========================
    # Map token saliency → chunks
    # ==========================
    chunk_saliency = []
    offset = 0

    for ct in chunk_texts:
        token_ids = _gen_tokenizer(ct, add_special_tokens=False)["input_ids"]
        n_tok = min(len(token_ids), len(saliency) - offset)

        if n_tok <= 0:
            chunk_saliency.append(0.0)
            continue

        chunk_saliency.append(float(saliency[offset:offset + n_tok].mean()))
        offset += n_tok

    n_keep = max(1, int(len(chunks) * retain_ratio))
    ranked = sorted(range(len(chunks)), key=lambda i: chunk_saliency[i], reverse=True)

    return [chunks[i] for i in sorted(ranked[:n_keep])]


print(f"Gradient selector loaded ({_GEN_MODEL_NAME}).")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


/opt/conda/lib/python3.10/site-packages/accelerate/utils/modeling.py:1365: UserWarning: Current model requires 603984384 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Gradient selector loaded (Qwen/Qwen2.5-3B-Instruct).


In [15]:
# ═══════════════════════════════════════════════════════════════════════════════
# Diverse Prompted Generation — RAG5
# ═══════════════════════════════════════════════════════════════════════════════

import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM

PERSONA_SYSTEM_ROLES = [
    "You are a concise medical assistant.",
    "You are a molecular geneticist focusing on mechanisms.",
    "You are a clinical researcher prioritizing strong evidence.",
    "You are a specialist in rare diseases and atypical presentations.",
    "You apply strict evidence-based medicine principles.",
    "You are a paediatric specialist.",
    "You are an epidemiologist focusing on population-level insights.",
]

PERSONA_TEMPLATES = [
    "Answer concisely based on the context.\nQuestion: {q}\nContext: {ctx}\nAnswer:",
    
    "Focus on molecular genetic mechanisms.\nQuestion: {q}\nContext: {ctx}\nAnswer:",
    
    "Prioritize clinically supported evidence.\nQuestion: {q}\nContext: {ctx}\nAnswer:",
    
    "Highlight rare conditions or atypical presentations if relevant.\nQuestion: {q}\nContext: {ctx}\nAnswer:",
    
    "Apply evidence-based reasoning.\nQuestion: {q}\nContext: {ctx}\nAnswer:",
    
    "Focus on paediatric relevance if applicable.\nQuestion: {q}\nContext: {ctx}\nAnswer:",
    
    "Consider population-level or epidemiological insights.\nQuestion: {q}\nContext: {ctx}\nAnswer:",
]

def _token_overlap(a: str, b: str) -> float:
    sa, sb = set(a.lower().split()), set(b.lower().split())
    return len(sa & sb) / len(sa | sb) if (sa and sb) else 0.0


# 🔥 cache model (important for speed)
_model_cache = {}

def _get_model(model_name):
    if model_name not in _model_cache:
        tok = AutoTokenizer.from_pretrained(
            model_name,
            trust_remote_code=True
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            torch_dtype=torch.float16,
            trust_remote_code=True
        )
        model.eval()
        _model_cache[model_name] = (tok, model)
    return _model_cache[model_name]


def diverse_prompted_generate(
    question: str,
    context: str,
    n_generations: int = 7,
    temperature: float = 0.8,
    model_name: str = "Qwen/Qwen2.5-3B-Instruct"
) -> str:
    """
    Generate N candidate answers using distinct expert persona prompts at high
    temperature, then return the candidate with the highest average token overlap
    with all other candidates (most semantically central / consensus answer).

    Personas (in order):
        1. Medical assistant (generic baseline)
        2. Molecular geneticist
        3. Clinical researcher
        4. Rare disease / comorbidity specialist
        5. Evidence-based medicine reviewer
        6. Paediatric specialist
        7. Epidemiologist

    Args:
        question      : User query.
        context       : Formatted retrieved context string.
        n_generations : Number of persona prompts to use (up to 7).
        temperature   : Sampling temperature (higher = more diverse).
        model_name    : HuggingFace generator model identifier.

    Returns:
        Consensus candidate answer string.
    """

    tok, model = _get_model(model_name)

    candidates = []

    for tmpl in PERSONA_TEMPLATES[:n_generations]:

        prompt = tmpl.format(q=question, ctx=context)

        for i, tmpl in enumerate(PERSONA_TEMPLATES[:n_generations]):
        
            system_role = PERSONA_SYSTEM_ROLES[i]
        
            prompt = tmpl.format(q=question, ctx=context)
        
            messages = [
                {"role": "system", "content": system_role},
                {"role": "user", "content": prompt},
            ]

        text = tok.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        enc = tok(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=2048   # 🔥 balanced speed + coverage
        ).to(model.device)

        with torch.no_grad():
            out = model.generate(
                **enc,
                max_new_tokens=64,
                do_sample=True,
                temperature=temperature
            )

        # 🔥 clean output (remove prompt part)
        gen_tokens = out[0][enc["input_ids"].shape[1]:]
        decoded = tok.decode(gen_tokens, skip_special_tokens=True).strip()

        candidates.append(decoded)

    # Select candidate with highest average pairwise token overlap
    scores = [
        np.mean([
            _token_overlap(c, candidates[j])
            for j in range(len(candidates)) if j != i
        ])
        for i, c in enumerate(candidates)
    ]

    return candidates[int(np.argmax(scores))]


print(f"Diverse prompted generation ready ({len(PERSONA_TEMPLATES)} personas defined).")

Diverse prompted generation ready (7 personas defined).


In [16]:
# ═══════════════════════════════════════════════════════════════════════════════
# K-Means Majority Vote Generation — RAG6
# ═══════════════════════════════════════════════════════════════════════════════

import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer

_SENT_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
_sent_model = SentenceTransformer(_SENT_MODEL_NAME)

# 🔥 cache generator model (important for speed)
_model_cache = {}

def _get_model(model_name):
    if model_name not in _model_cache:
        tok = AutoTokenizer.from_pretrained(
            model_name,
            trust_remote_code=True
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            torch_dtype=torch.float16,
            trust_remote_code=True
        )
        model.eval()
        _model_cache[model_name] = (tok, model)
    return _model_cache[model_name]


def kmeans_majority_generate(
    question: str,
    context: str,
    n_generations: int = 7,
    temperature: float = 0.8,
    n_clusters: int = 3,
    model_name: str = "Qwen/Qwen2.5-3B-Instruct"
) -> str:
    """
    Generate N candidate answers, embed them, cluster with K-Means, and return
    the response closest to the centroid of the largest (majority) cluster.

    Algorithm:
        1. Generate N answers by sampling the generator at `temperature`.
        2. Encode each answer with a sentence transformer (all-MiniLM-L6-v2).
        3. Run K-Means (k = n_clusters) on the embedding matrix.
        4. Identify the majority cluster (most members = plurality vote).
        5. Return the member closest to that cluster's centroid.

    Advantage over RAG5: operates in embedding space, so it is invariant to
    surface-level paraphrasing and more robust than token-overlap voting.

    Args:
        question      : User query.
        context       : Formatted retrieved context string.
        n_generations : Number of candidate answers to sample.
        temperature   : Sampling temperature.
        n_clusters    : K for K-Means (should be < n_generations).
        model_name    : HuggingFace generator model identifier.

    Returns:
        Answer string closest to the majority cluster centroid.
    """
    tok, model = _get_model(model_name)

    prompt = (
        f"Answer the question based only on the context.\n"
        f"Question: {question}\nContext: {context}\nAnswer:"
    )

    # 🔥 Use chat template (important for Qwen)
    messages = [
        {"role": "system", "content": "You are a biomedical expert assistant."},
        {"role": "user", "content": prompt},
    ]

    text = tok.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    enc = tok(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=2048   # 🔥 balanced
    ).to(model.device)

    input_len = enc["input_ids"].shape[1]

    # Step 1 — Generate N candidates
    candidates = []
    with torch.no_grad():
        for _ in range(n_generations):
            out = model.generate(
                **enc,
                max_new_tokens=64,
                do_sample=True,
                temperature=temperature
            )

            # 🔥 remove prompt portion
            gen_tokens = out[0][input_len:]
            decoded = tok.decode(gen_tokens, skip_special_tokens=True).strip()

            candidates.append(decoded)

    # Step 2 — Embed candidates
    embeddings = _sent_model.encode(candidates, convert_to_numpy=True)  # (N, D)

    # Step 3 — K-Means clustering
    k      = min(n_clusters, len(candidates))
    km     = KMeans(n_clusters=k, random_state=42, n_init="auto")
    labels = km.fit_predict(embeddings)

    # Step 4 — Identify majority cluster and select centroid-closest member
    majority = int(np.argmax(np.bincount(labels)))
    members  = np.where(labels == majority)[0]
    dists    = np.linalg.norm(
        embeddings[members] - km.cluster_centers_[majority],
        axis=1
    )

    return candidates[int(members[np.argmin(dists)])]


print(f"K-Means majority vote generator ready (sentence encoder: {_SENT_MODEL_NAME}).")
print("\n✓ Section 0 complete — all helpers loaded. Proceed to Section 1.")

K-Means majority vote generator ready (sentence encoder: sentence-transformers/all-MiniLM-L6-v2).

✓ Section 0 complete — all helpers loaded. Proceed to Section 1.


---
# Section 1: RAG2 — Chunking Strategy Comparison

**Goal:** Determine the best chunking strategy to carry forward into Sections 2–5.

Both strategies use the same **hybrid retrieval** pipeline (dense + BM25).
The only variable is how the corpus was chunked at index build time.

| Strategy                      | Description                                                | Expected behaviour                                                    |
| ----------------------------- | ---------------------------------------------------------- | --------------------------------------------------------------------- |
| `section_400token`            | Section-aware, ~400 tokens                                 | Preserves semantic boundaries; better for multi-sentence answers      |
| `fixed_500char`               | Fixed 500 characters                                       | Simpler; may split mid-sentence, losing context                       |
| `section_contextual_400token` | Section-aware + LLM-generated contextual summary per chunk | Enhances semantic richness; improves retrieval alignment with queries |


**Hypothesis:** `section_400token` will outperform fixed_500char because section boundaries represent natural semantic units in biomedical literature, helping preserve related information within a single chunk.

Furthermore, `section_contextual_400token` is expected to achieve the best performance overall, as augmenting each chunk with an LLM-generated summary provides additional semantic signals. This enrichment can improve both dense and sparse retrieval by making relevant chunks more distinguishable and better aligned with user queries.

In [ ]:
print("=" * 60)
print("Section 1: RAG2 — chunking strategy comparison")
print("=" * 60)

print("\n[1/3] RAG2 with section_400token ...")
df_s1_section = run_rag_loop(
    label="RAG2 (section_400token)",
    dense_fn=dense_fn_section,
    bm25_fn=bm25_fn_section,
    chunk_strategy="section_400token",
    use_bm25=True,
    output_csv="rag2_section_400token.csv"
)

print("\n[2/3] RAG2 with fixed_500char ...")
df_s1_fixed = run_rag_loop(
    label="RAG2 (fixed_500char)",
    dense_fn=dense_fn_fixed,
    bm25_fn=bm25_fn_fixed,
    chunk_strategy="fixed_500char",
    use_bm25=True,
    output_csv="rag2_fixed_500char.csv"
)

print("\n[3/3] RAG2 with section_contextual_400token ...")
df_s1_contextual = run_rag_loop(
    label="RAG2 (section_contextual_400token)",
    dense_fn=dense_fn_contextual,
    bm25_fn=bm25_fn_contextual,
    chunk_strategy="section_contextual_400token",
    use_bm25=True,
    output_csv="rag2_section_contextual_400token.csv"
)

Section 1: RAG2 — chunking strategy comparison

[1/3] RAG2 with section_400token ...
  [RAG2 (section_400token)] 1/50


In [ ]:
BEST_CHUNK_STRATEGY = "section_contextual_400token"
dense_fn_best = dense_fn_contextual
bm25_fn_best  = bm25_fn_contextual

In [ ]:
'''print("\n── Section 1 Results ──")
s1_cmp = compare({
    "RAG2 (section_400token)": df_s1_section,
    "RAG2 (fixed_500char)":    df_s1_fixed,
})
print("Expected: section_400token >= fixed_500char")

# Auto-select best strategy and wire up retrievers for Sections 2–5
acc_section = score_results(df_s1_section, ref_map)
acc_fixed   = score_results(df_s1_fixed,   ref_map)

BEST_CHUNK_STRATEGY = "section_400token" if acc_section >= acc_fixed else "fixed_500char"
dense_fn_best = dense_fn_section if BEST_CHUNK_STRATEGY == "section_400token" else dense_fn_fixed
bm25_fn_best  = bm25_fn_section  if BEST_CHUNK_STRATEGY == "section_400token" else bm25_fn_fixed

print(f"\n✓ Best chunking strategy: {BEST_CHUNK_STRATEGY}")
print(f"  This will be used in all subsequent sections.")'''

## Section 1 — Export Eval CSVs

Writes eval-ready CSVs (`input`, `actual_output`, `retrieval_context`) that can be
loaded directly into the `medical-rag-eval` notebook.

**To evaluate in the eval notebook**, set:
```python
# Replace the csv_data block with:
# (or just point the reader at the file path printed below)
with open("eval_s1_section_400token.csv", "r") as f: ...
```

In [ ]:
'''print("── Section 1: Eval CSV Export ──")
export_for_eval(df_s1_section, "eval_s1_section_400token.csv")
export_for_eval(df_s1_fixed,   "eval_s1_fixed_500char.csv")
print("\nLoad either file into medical-rag-eval.ipynb to run GEval metrics.")'''

---
# Section 2: RAG1 (Dense-Only) vs RAG2 (Hybrid)

**Goal:** Confirm that adding BM25 sparse retrieval improves performance over dense-only.

Both RAGs use the best chunk strategy selected in Section 1.

| RAG | Retrieval | Generation |
|-----|-----------|------------|
| RAG1 | Dense-only (PubMedBERT cosine similarity) | Standard (flan-t5-small, greedy) |
| RAG2 | Hybrid (Dense + BM25 score fusion) | Standard (flan-t5-small, greedy) |

**Hypothesis:** RAG2 > RAG1. BM25 captures exact biomedical term matches (e.g. gene
names, drug identifiers, rare disease names) that dense embeddings tend to miss
because they match semantically similar but lexically different passages.

In [ ]:
print("=" * 60)
print(f"Section 2: RAG1 vs RAG2  [chunk: {BEST_CHUNK_STRATEGY}]")
print("=" * 60)

print("\n[1/2] RAG1 — dense-only retrieval ...")
df_s2_rag1 = run_rag_loop(
    label="RAG1 (dense-only)",
    dense_fn=dense_fn_best, bm25_fn=bm25_fn_best,
    chunk_strategy=BEST_CHUNK_STRATEGY,
    use_bm25=False,
    output_csv="rag1_eval.csv"
)

print("\n[2/2] RAG2 — hybrid retrieval ...")
df_s2_rag2 = run_rag_loop(
    label="RAG2 (hybrid)",
    dense_fn=dense_fn_best, bm25_fn=bm25_fn_best,
    chunk_strategy=BEST_CHUNK_STRATEGY,
    use_bm25=True,
    output_csv="rag2_eval.csv"
)

In [ ]:
'''print("\n── Section 2 Results ──")
s2_cmp = compare({
    "RAG1 (dense-only)": df_s2_rag1,
    "RAG2 (hybrid)":     df_s2_rag2,
})
print("Expected: RAG2 > RAG1")'''

## Section 2 — Export Eval CSVs

In [ ]:
'''print("── Section 2: Eval CSV Export ──")
export_for_eval(df_s2_rag1, "eval_s2_rag1.csv")
export_for_eval(df_s2_rag2, "eval_s2_rag2.csv")
print("\nLoad either file into medical-rag-eval.ipynb to run GEval metrics.")'''

---
# Section 3: RAG2 (Hybrid) vs RAG3 (Hybrid + Reranking)

**Goal:** Test whether cross-encoder reranking improves the precision of context
passed to the generator.

| RAG | Retrieval | Reranking | Generation |
|-----|-----------|-----------|------------|
| RAG2 | Hybrid (Dense + BM25) | ✗ | Standard |
| RAG3 | Hybrid (Dense + BM25) | ✓ Cross-encoder | Standard |

**How reranking works:** Hybrid retrieval returns 20 candidates via score fusion.
A cross-encoder (`ms-marco-MiniLM-L-6-v2`) re-scores each (query, chunk) pair
*jointly* — query and document tokens attend to each other — giving more accurate
relevance estimates than the bi-encoder. The top 10 chunks are passed to the generator.

**Hypothesis:** RAG3 > RAG2. Reranking filters out false positives from the
hybrid fusion step, giving the generator a more precisely relevant context window.

In [ ]:
print("=" * 60)
print(f"Section 3: RAG2 vs RAG3  [chunk: {BEST_CHUNK_STRATEGY}]")
print("=" * 60)

# RAG2 reused from Section 2 (df_s2_rag2)
print("\n[1/1] RAG3 — hybrid + reranking ...")
df_s3_rag3 = run_rag_loop(
    label="RAG3 (+ reranking)",
    dense_fn=dense_fn_best, bm25_fn=bm25_fn_best,
    chunk_strategy=BEST_CHUNK_STRATEGY,
    use_bm25=True, use_reranking=True,
    output_csv="rag3_eval.csv"
)

In [ ]:
'''print("\n── Section 3 Results ──")
s3_cmp = compare({
    "RAG2 (hybrid)":    df_s2_rag2,
    "RAG3 (+ reranking)": df_s3_rag3,
})
print("Expected: RAG3 > RAG2")'''

## Section 3 — Export Eval CSVs

In [ ]:
'''print("── Section 3: Eval CSV Export ──")
export_for_eval(df_s3_rag3, "eval_s3_rag3.csv")
print("\nLoad into medical-rag-eval.ipynb to run GEval metrics.")'''

---
# Section 4: RAG3 (Reranking) vs RAG4 (+ Gradient-Based Selection)

**Goal:** Test whether pruning low-salience context chunks further improves
answer accuracy after reranking.

| RAG | Retrieval | Reranking | Context Pruning | Generation |
|-----|-----------|-----------|-----------------|------------|
| RAG3 | Hybrid | ✓ Cross-encoder | ✗ | Standard |
| RAG4 | Hybrid | ✓ Cross-encoder | ✓ Gradient saliency | Standard |

**How gradient-based selection works:** After reranking, 10 chunks are concatenated
as context. Gradient-based selection runs a forward+backward pass through the generator
and computes per-token saliency:

$$s_i = \left\|\frac{\partial \mathcal{L}}{\partial e_i}\right\|_2$$

Chunks are ranked by mean token saliency and the bottom 40% are dropped
(`retain_ratio = 0.6`), leaving ~6 high-signal chunks for the generator.

**Trade-off:** Requires one additional forward+backward pass per question
(~2× inference time vs. RAG3).

**Hypothesis:** RAG4 > RAG3 (modest gain). Removing tangential context reduces
the chance of the generator being misled by irrelevant passages.

In [ ]:
print("=" * 60)
print(f"Section 4: RAG3 vs RAG4  [chunk: {BEST_CHUNK_STRATEGY}]")
print("=" * 60)

# RAG3 reused from Section 3 (df_s3_rag3)
print("\n[1/1] RAG4 — hybrid + reranking + gradient selection ...")
df_s4_rag4 = run_rag_loop(
    label="RAG4 (+ gradient selection)",
    dense_fn=dense_fn_best, bm25_fn=bm25_fn_best,
    chunk_strategy=BEST_CHUNK_STRATEGY,
    use_bm25=True, use_reranking=True, use_gradient=True,
    output_csv="rag4_eval.csv"
)

In [ ]:
'''print("\n── Section 4 Results ──")
s4_cmp = compare({
    "RAG3 (+ reranking)":        df_s3_rag3,
    "RAG4 (+ gradient selection)": df_s4_rag4,
})
print("Expected: RAG4 > RAG3 (small improvement)")'''

## Section 4 — Export Eval CSVs

In [ ]:
'''print("── Section 4: Eval CSV Export ──")
export_for_eval(df_s4_rag4, "eval_s4_rag4.csv")
print("\nLoad into medical-rag-eval.ipynb to run GEval metrics.")'''

---
# Section 5: RAG4 vs RAG5 (Diverse Prompts) vs RAG6 (K-Means Vote)

**Goal:** Compare two advanced generation strategies against the RAG4 baseline.
All three share the same retrieval pipeline (hybrid + reranking + gradient selection).
The only difference is the generation strategy.

| RAG | Retrieval | Generation strategy |
|-----|-----------|--------------------|
| RAG4 | Hybrid + Reranking + Gradient | Single prompt, greedy decoding |
| RAG5 | Hybrid + Reranking + Gradient | 7 expert persona prompts, temperature sampling, token-overlap vote |
| RAG6 | Hybrid + Reranking + Gradient | N samples, sentence embeddings, K-Means clustering, centroid vote |

---

### RAG5 — Diverse Prompted Generation
Uses 7 expert persona prompts (*medical assistant, geneticist, clinical researcher,
rare-disease specialist, EBM reviewer, paediatric specialist, epidemiologist*)
each sampled at `temperature = 0.8`. The final answer is the candidate with the
highest average **token-overlap** with all others (most semantically central).

**Hypothesis:** Diverse prompts surface rare domain knowledge that a single generic
prompt underweights, improving recall on specialist medical questions.

---

### RAG6 — K-Means Majority Vote
Generates N = 7 responses at `temperature = 0.8`, encodes them with
`all-MiniLM-L6-v2`, clusters with K-Means (k = 3), and selects the response
closest to the centroid of the largest cluster.

Unlike RAG5's token-overlap vote (lexical), K-Means operates in embedding space
and is robust to paraphrasing — semantically equivalent answers cluster together
regardless of surface wording.

**Hypothesis:** RAG6 > RAG5 > RAG4 overall. Embedding-space clustering is a more
principled form of majority voting and better handles lexical variation.

In [ ]:
print("=" * 60)
print(f"Section 5: RAG4 vs RAG5 vs RAG6  [chunk: {BEST_CHUNK_STRATEGY}]")
print("=" * 60)

# RAG4 reused from Section 4 (df_s4_rag4)

print("\n[1/2] RAG5 — diverse prompted generation ...")
df_s5_rag5 = run_rag_loop(
    label="RAG5 (diverse prompts)",
    dense_fn=dense_fn_best, bm25_fn=bm25_fn_best,
    chunk_strategy=BEST_CHUNK_STRATEGY,
    use_bm25=True, use_reranking=True, use_gradient=True, use_diverse=True,
    output_csv="rag5_eval.csv"
)

print("\n[2/2] RAG6 — K-Means majority vote ...")
df_s5_rag6 = run_rag_loop(
    label="RAG6 (k-means vote)",
    dense_fn=dense_fn_best, bm25_fn=bm25_fn_best,
    chunk_strategy=BEST_CHUNK_STRATEGY,
    use_bm25=True, use_reranking=True, use_gradient=True, use_kmeans=True,
    output_csv="rag6_eval.csv"
)

In [ ]:
'''print("\n── Section 5 Results ──")
s5_cmp = compare({
    "RAG4 (+ gradient selection)": df_s4_rag4,
    "RAG5 (diverse prompts)":      df_s5_rag5,
    "RAG6 (k-means vote)":         df_s5_rag6,
})
print("Expected: RAG6 > RAG5 > RAG4")'''

## Section 5 — Export Eval CSVs

In [ ]:
'''print("── Section 5: Eval CSV Export ──")
export_for_eval(df_s5_rag5, "eval_s5_rag5.csv")
export_for_eval(df_s5_rag6, "eval_s5_rag6.csv")
print("\nLoad either file into medical-rag-eval.ipynb to run GEval metrics.")'''

---
# Final Summary — All Sections

Run this cell after completing all sections for the full cross-experiment comparison
table. Results are saved to `rag_full_summary.csv`.

In [ ]:
print("=" * 60)
print("FULL RAG COMPARISON SUMMARY")
print(f"Best chunk strategy (Section 1): {BEST_CHUNK_STRATEGY}")
print("=" * 60)

all_results = {
    "S1 — RAG2 (section_400token)": df_s1_section,
    "S1 — RAG2 (fixed_500char)":    df_s1_fixed,
    "S1 — RAG2 (section_contextual_400token)": df_s1_contextual,
    "S2 — RAG1 (dense-only)":       df_s2_rag1,
    "S2 — RAG2 (hybrid)":           df_s2_rag2,
    "S3 — RAG3 (+ reranking)":      df_s3_rag3,
    "S4 — RAG4 (+ gradient sel.)":  df_s4_rag4,
    "S5 — RAG5 (diverse prompts)": df_s5_rag5,
    "S5 — RAG6 (k-means vote)":     df_s5_rag6,
}

summary_rows = []
for label, df in all_results.items():
    acc = score_results(df, ref_map)
    summary_rows.append({
        "Variant":        label,
        "n":              len(df),
        "Exact Match Acc": f"{acc:.4f}" if not pd.isna(acc) else "N/A",
    })

final_df = pd.DataFrame(summary_rows)
print(final_df.to_string(index=False))

final_df.to_csv(RAG_DIR /"rag_full_summary.csv", index=False)
print("\nFull summary saved → rag_full_summary.csv")

# ── Combined eval CSV (all RAGs, eval-notebook format) ────────────────────────
# Stacks all RAG outputs into one file so you can evaluate all variants in a
# single medical-rag-eval.ipynb run. The 'rag_variant' column lets you filter
# or group results after scoring.
combined_eval = pd.concat(
    [df[["input", "actual_output", "retrieval_context", "rag_variant"]]
     for df in all_results.values()],
    ignore_index=True
)
combined_eval.to_csv(RAG_DIR / "eval_all_rags.csv", index=False, encoding="utf-8")
print("Combined eval CSV saved → eval_all_rags.csv")
print("  Columns: input | actual_output | retrieval_context | rag_variant")
print("  Load into medical-rag-eval.ipynb (strip 'rag_variant' col or keep for grouping).")

final_df